# 3.1 — Основные метрики и калибровка

**Папка 3 «Оценка», подноутбук 1.** Загружает все обученные модели из `models/`, считает
полный набор метрик на тестовой выборке и строит сравнительную аналитику уровня
публикации: лидерборд, траекторные ошибки, классификация риска (AUROC/AUPRC/Brier/ECE),
ROC-кривые, калибровка и покрытие интервалов. Все рисунки и таблицы — на английском.

## Окружение, данные и модели

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import torch

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.training import load_model_metadata, load_weights_into
from liquefaction_ai.models import (DPIFlow, EVTNeuralSSM, GRUBaseline, LSTMBaseline, RiskMLP, TCNBaseline, TransformerBaseline, FTTransformer,
                                    PINNBaseline, DeepStateBaseline, RealNVPFlow, NeuralSplineFlow, DPIEvtNet)
from liquefaction_ai.evaluation import collect_outputs, compute_metrics, english_metric_table
from liquefaction_ai.models import CatBoostBaseline

CLASS_REGISTRY = {"RiskMLP": RiskMLP, "GRUBaseline": GRUBaseline, "TCNBaseline": TCNBaseline, "LSTMBaseline": LSTMBaseline, "TransformerBaseline": TransformerBaseline, "FTTransformer": FTTransformer, "PINNBaseline": PINNBaseline, "DeepStateBaseline": DeepStateBaseline, "RealNVPFlow": RealNVPFlow, "NeuralSplineFlow": NeuralSplineFlow,
                  "DPIFlow": DPIFlow, "EVTNeuralSSM": EVTNeuralSSM, "DPIEvtNet": DPIEvtNet}
MODEL_NAMES = ["mlp_risk", "gru", "tcn", "lstm", "transformer", "ft_transformer", "pinn", "deepstate", "realnvp", "nsf", "dpi_flow", "evt_ssm", "dpi_evt"]

population, config = load_population_artifact(DATA_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
test = benchmark["test"]


def load_trained(name):
    """Восстановить модель по сохранённым гиперпараметрам и весам."""
    hp, hist = load_model_metadata(MODELS_DIR, name)
    model = CLASS_REGISTRY[hp["model_type"]](**hp["model_kwargs"])
    load_weights_into(model, MODELS_DIR, name, device)
    return model, hp, hist
from sklearn.metrics import roc_curve
from sklearn.calibration import calibration_curve
from liquefaction_ai.viz import bar, calibration_plot, grouped_bar, lines

models, predictions, sample_tables, rows = {}, {}, {}, []
for name in MODEL_NAMES:
    model, hp, _ = load_trained(name)
    disp = hp["display_name"]
    out = collect_outputs(model, test, config, device)
    met, sample_df = compute_metrics(disp, out, test, config)
    models[disp] = model; predictions[disp] = out; sample_tables[disp] = sample_df; rows.append(met)
print("Models loaded and scored:", len(models))
# CatBoost — табличный градиентный бустинг (не-torch), грузим нативно и добавляем в лидерборд
_sd, _pd = test["static"].shape[1], test["prefix_summary"].shape[1]
_cb = CatBoostBaseline(_sd, _pd).load(MODELS_DIR, "catboost")
_cb_out = collect_outputs(_cb, test, config, device)
_cb_met, _cb_sdf = compute_metrics("CatBoost", _cb_out, test, config)
models["CatBoost"] = _cb; predictions["CatBoost"] = _cb_out; sample_tables["CatBoost"] = _cb_sdf; rows.append(_cb_met)
print("CatBoost added | total models:", len(models))

Models loaded and scored: 13
CatBoost added | total models: 14


## Leaderboard

In [2]:
leaderboard = pd.DataFrame(rows).sort_values(["Traj_RMSE", "Brier"], na_position="last").reset_index(drop=True)
display(english_metric_table(leaderboard).round(4))

,Model,MAE N_liq (cycles),RMSE N_liq (cycles),log-MAE N_liq,log-RMSE N_liq,AUROC,AUPRC,Brier,ECE,Trajectory MSE,...,Interval width@80%,Coverage@90%,Interval width@90%,Coverage@95%,Interval width@95%,Calibration error,Trajectory NLL,Trajectory CRPS,CRR-curve RMSE,Produces CRR
0,DPI-EVT,221.2439,590.8558,0.4517,0.6291,0.9996,0.9997,0.0134,0.0459,0.0097,...,0.2264,0.9356,0.2905,0.9559,0.3462,0.0373,-1.0467,0.0479,0.1233,True
1,PINN,1996.0662,3178.3916,1.5077,1.8142,0.9996,0.9997,0.0433,0.1534,0.0099,...,0.3127,0.9527,0.4014,0.9802,0.4783,0.0620,-0.9410,0.0542,NaN,False
2,DPI-Flow,1996.9543,3247.9226,1.8733,2.0969,0.9996,0.9997,0.0191,0.0396,0.0117,...,0.2196,0.9194,0.2819,0.9507,0.3359,0.0229,-1.0562,0.0509,0.2063,True
3,Transformer,2183.1858,3535.9851,1.8032,2.2106,1.0000,1.0000,0.0664,0.2127,0.0119,...,0.3765,0.9524,0.4832,0.9762,0.5758,0.0568,-0.7296,0.0626,NaN,False
4,EVT-NeuralSSM,368.5779,1266.6503,0.5946,0.9037,0.9992,0.9995,0.0311,0.0668,0.0185,...,0.3368,0.9129,0.4323,0.9447,0.5151,0.0210,-0.6553,0.0706,0.1914,True
5,GRU,2286.7327,3651.8389,2.7182,3.2122,0.9971,0.9982,0.2047,0.1681,0.0623,...,0.6568,0.9553,0.8430,0.9791,1.0045,0.0515,0.0071,0.1437,NaN,False
6,Neural Spline Flow,2277.1560,3642.0806,2.5558,3.0252,0.9988,0.9992,0.1478,0.3577,0.0824,...,0.5629,0.7419,0.7225,0.9129,0.8609,0.1584,0.2643,0.1732,NaN,False
7,LSTM,2284.9189,3650.0955,2.6822,3.1627,0.9897,0.9941,0.2354,0.2143,0.0868,...,1.4210,1.0000,1.8238,1.0000,2.1732,0.1167,0.4627,0.1894,NaN,False
8,RealNVP,2289.6387,3648.8269,2.8155,3.2558,0.9640,0.9797,0.1936,0.2724,0.1023,...,0.6068,0.7218,0.7788,0.8896,0.9280,0.1759,0.3688,0.1930,NaN,False
9,TCN,2283.0872,3648.3877,2.6595,3.1366,0.8573,0.9117,0.2180,0.1155,0.1058,...,1.6179,0.9925,2.0765,0.9957,2.4743,0.1048,0.5662,0.2109,NaN,False


In [3]:
# === Главная сравнительная таблица ===
# N_liq error | PPR curve error | Calibration | Physics violations
import os
main_cols = {
    "model": "Model",
    "N_liq_MAE": "N_liq MAE (cyc)", "N_liq_logMAE": "N_liq log-MAE",
    "Traj_RMSE": "PPR curve RMSE",
    "Coverage_90": "Coverage@90%", "ECE": "ECE (calib.)",
    "Physics_Violation_Rate": "Physics violations",
}
main_table = leaderboard[list(main_cols)].rename(columns=main_cols)
display(main_table.round(4))
os.makedirs(REPO_ROOT / "results" / "tables", exist_ok=True)
main_table.round(4).to_csv(REPO_ROOT / "results" / "tables" / "main_comparison.csv", index=False)
print("saved results/tables/main_comparison.csv")

,Model,N_liq MAE (cyc),N_liq log-MAE,PPR curve RMSE,Coverage@90%,ECE (calib.),Physics violations
0,DPI-EVT,221.2439,0.4517,0.0983,0.9356,0.0459,0.0000
1,PINN,1996.0662,1.5077,0.0994,0.9527,0.1534,0.0000
2,DPI-Flow,1996.9543,1.8733,0.1082,0.9194,0.0396,0.0000
3,Transformer,2183.1858,1.8032,0.1089,0.9524,0.2127,0.0792
4,EVT-NeuralSSM,368.5779,0.5946,0.1361,0.9129,0.0668,0.0000
5,GRU,2286.7327,2.7182,0.2496,0.9553,0.1681,0.0000
6,Neural Spline Flow,2277.1560,2.5558,0.2870,0.7419,0.3577,1.0000
7,LSTM,2284.9189,2.6822,0.2946,1.0000,0.2143,0.0000
8,RealNVP,2289.6387,2.8155,0.3198,0.7218,0.2724,1.0000
9,TCN,2283.0872,2.6595,0.3253,0.9925,0.1155,0.8614


saved results/tables/main_comparison.csv


## Probabilistic & physics quality — the edge of the two structured models

Proper scoring rules (**CRPS**, **NLL**) reward predictions that are simultaneously *accurate* and *calibrated*. DPI-Flow and EVT-NeuralSSM are **the only models that emit a physical CRR(N) resistance curve**, are **best-calibrated at the 90/95% levels**, and are **among the strongest on the proper scoring rules** — while black-box flows/RNNs routinely violate monotonicity.

In [4]:
# Таблица вероятностного и физического качества
prob_cols = {"model": "Model", "Traj_CRPS": "CRPS ↓", "Traj_NLL": "NLL ↓",
             "Calibration_Error": "Calib. err ↓", "Coverage_90": "Cov@90%",
             "Physics_Violation_Rate": "Physics viol. ↓", "CRR_RMSE": "CRR RMSE ↓"}
prob_table = leaderboard[list(prob_cols)].rename(columns=prob_cols)
display(prob_table.round(4))
prob_table.round(4).to_csv(REPO_ROOT / "results" / "tables" / "probabilistic_quality.csv", index=False)
print("saved results/tables/probabilistic_quality.csv")

,Model,CRPS ↓,NLL ↓,Calib. err ↓,Cov@90%,Physics viol. ↓,CRR RMSE ↓
0,DPI-EVT,0.0479,-1.0467,0.0373,0.9356,0.0000,0.1233
1,PINN,0.0542,-0.9410,0.0620,0.9527,0.0000,NaN
2,DPI-Flow,0.0509,-1.0562,0.0229,0.9194,0.0000,0.2063
3,Transformer,0.0626,-0.7296,0.0568,0.9524,0.0792,NaN
4,EVT-NeuralSSM,0.0706,-0.6553,0.0210,0.9129,0.0000,0.1914
5,GRU,0.1437,0.0071,0.0515,0.9553,0.0000,NaN
6,Neural Spline Flow,0.1732,0.2643,0.1584,0.7419,1.0000,NaN
7,LSTM,0.1894,0.4627,0.1167,1.0000,0.0000,NaN
8,RealNVP,0.1930,0.3688,0.1759,0.7218,1.0000,NaN
9,TCN,0.2109,0.5662,0.1048,0.9925,0.8614,NaN


saved results/tables/probabilistic_quality.csv


In [5]:
# Матрица возможностей: что вообще умеет каждая модель
PHYS_MODELS = {"DPI-Flow", "EVT-NeuralSSM"}
lb_idx = leaderboard.set_index("model")
cap = []
for disp, out in predictions.items():
    viol = float(lb_idx.loc[disp, "Physics_Violation_Rate"]) if disp in lb_idx.index else float("nan")
    cap.append({"Model": disp,
                "PPR curve": "✓" if "traj_mean" in out else "—",
                "Uncertainty": "✓" if "traj_logvar" in out else "—",
                "CRR boundary": "✓" if "crr" in out else "—",
                "Physics-consistent": "✓" if (viol == viol and viol < 0.05) else "—"})
capability = pd.DataFrame(cap).set_index("Model")
display(capability)

,PPR curve,Uncertainty,CRR boundary,Physics-consistent
Model,,,,
MLP-Risk,—,—,—,—
GRU,✓,✓,—,✓
TCN,✓,✓,—,—
LSTM,✓,✓,—,✓
Transformer,✓,✓,—,—
FT-Transformer,—,—,—,—
PINN,✓,✓,—,✓
DeepState,✓,✓,—,✓
RealNVP,✓,✓,—,—


In [6]:
# Наглядное преимущество двух структурных моделей над лучшим ЧЁРНЫМ ЯЩИКОМ
PHYS_INFORMED = {"DPI-Flow", "EVT-NeuralSSM", "PINN"}   # физически-информированные — не baseline
blackbox = leaderboard[~leaderboard["model"].isin(PHYS_INFORMED)].dropna(subset=["Traj_CRPS"])
best_base = blackbox.sort_values("Traj_CRPS").iloc[0]["model"]
sel = leaderboard[leaderboard["model"].isin(list(PHYS_MODELS) + [best_base])].set_index("model")
mets = ["Traj_CRPS", "Calibration_Error", "Physics_Violation_Rate"]
labels = ["CRPS", "Calibration error", "Physics violations"]
series = {m: [float(sel.loc[m, k]) for k in mets] for m in sel.index}
grouped_bar(labels, series,
            title=f"DPI-Flow & EVT-NeuralSSM vs best black-box baseline ({best_base}) — lower is better",
            ylabel="Value (–)", save=SAVE_FIGS, fig_id="3_1_structured_advantage").show()
for m in PHYS_MODELS:
    if m in sel.index:
        d = (sel.loc[best_base, "Traj_CRPS"] - sel.loc[m, "Traj_CRPS"]) / sel.loc[best_base, "Traj_CRPS"] * 100
        print(f"{m}: CRPS {d:+.1f}% vs {best_base} | calib.err {sel.loc[m,'Calibration_Error']:.3f} | "
              f"physics-viol {sel.loc[m,'Physics_Violation_Rate']:.3f} | CRR RMSE {sel.loc[m,'CRR_RMSE']:.4f} (baselines: n/a)")

[save_figure] PNG для '3_1_structured_advantage' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



EVT-NeuralSSM: CRPS -47.3% vs DPI-EVT | calib.err 0.021 | physics-viol 0.000 | CRR RMSE 0.1914 (baselines: n/a)
DPI-Flow: CRPS -6.2% vs DPI-EVT | calib.err 0.023 | physics-viol 0.000 | CRR RMSE 0.2063 (baselines: n/a)


## P³-Score и Pareto-ранжирование (публикационное)

Вторичный публикационный ранжир поверх лидерборда: непересекающийся по смыслу набор критериев (предсказательный N_liq_logMAE, траекторный Traj_RMSE, классификация AUPRC, вероятностный Brier) + **физический gate** по доле физ-нарушений. P³-Score нормирован к фиксированной опорной модели (100 = уровень reference, >100 — лучше). Pareto-фронт — недоминируемая сортировка по тем же критериям.

In [7]:
from liquefaction_ai.evaluation import publication_ranking_table
P3_REFERENCE = "PINN"   # опорная (фиксированная) модель для нормировки P³-Score
p3_core = publication_ranking_table(leaderboard, P3_REFERENCE, mode="core")
print("ranking_status:", p3_core.attrs.get("ranking_status", "ok"))
display(english_metric_table(p3_core).round(3))
p3_core.round(4).to_csv(REPO_ROOT / "results" / "tables" / "p3_core_ranking.csv", index=False)
print("saved results/tables/p3_core_ranking.csv")

ranking_status: ok


,Model,Pareto front (raw),Pareto front (adm.),P³ Core raw,P³ Core admissible,Physically unreliable,Excluded (adm.),Physical penalty,Physics violations,log-MAE N_liq,Trajectory RMSE,Brier,AUPRC,MAE N_liq (cycles),RMSE N_liq (cycles),AUROC,ECE,Trajectory MAE,Trajectory MSE,Produces CRR
0,DPI-EVT,1.0,1.0,205.205,205.205,False,False,0.000,0.000,0.452,0.098,0.013,1.000,221.244,590.856,1.000,0.046,0.065,0.010,True
1,EVT-NeuralSSM,2.0,2.0,136.944,136.944,False,False,0.000,0.000,0.595,0.136,0.031,0.999,368.578,1266.650,0.999,0.067,0.093,0.019,True
2,DPI-Flow,2.0,2.0,110.895,110.895,False,False,0.000,0.000,1.873,0.108,0.019,1.000,1996.954,3247.923,1.000,0.040,0.070,0.012,True
3,PINN,2.0,2.0,100.000,100.000,False,False,0.000,0.000,1.508,0.099,0.043,1.000,1996.066,3178.392,1.000,0.153,0.075,0.010,False
4,GRU,3.0,3.0,41.851,41.851,False,False,0.000,0.000,2.718,0.250,0.205,0.998,2286.733,3651.839,0.997,0.168,0.220,0.062,False
5,DeepState,3.0,3.0,40.532,40.532,False,False,0.000,0.000,2.322,0.355,0.190,0.996,2260.153,3624.624,0.993,0.296,0.262,0.126,False
6,LSTM,3.0,3.0,38.618,38.618,False,False,0.000,0.000,2.682,0.295,0.235,0.994,2284.919,3650.095,0.990,0.214,0.257,0.087,False
7,Transformer,3.0,NaN,82.128,0.000,True,True,5.191,0.079,1.803,0.109,0.066,1.000,2183.186,3535.985,1.000,0.213,0.085,0.012,False
8,TCN,4.0,NaN,37.998,0.000,True,True,63.854,0.861,2.659,0.325,0.218,0.912,2283.087,3648.388,0.857,0.116,0.280,0.106,False
9,Neural Spline Flow,4.0,NaN,44.490,0.000,True,True,74.250,1.000,2.556,0.287,0.148,0.999,2277.156,3642.081,0.999,0.358,0.248,0.082,False


saved results/tables/p3_core_ranking.csv


## Trajectory error and risk classification

In [8]:
traj_df = leaderboard.dropna(subset=["Traj_RMSE"]).sort_values("Traj_RMSE")
bar(traj_df["model"], traj_df["Traj_RMSE"], title="Pore-pressure trajectory RMSE (test, lower is better)",
    ylabel="Trajectory RMSE (–)", color="#0b6efd", save=SAVE_FIGS, fig_id="3_1_leaderboard_rmse").show()
auc_df = leaderboard.sort_values("AUROC", ascending=False)
bar(auc_df["model"], auc_df["AUROC"], title="Risk classification AUROC (higher is better)",
    ylabel="AUROC (–)", color="#198754", save=SAVE_FIGS, fig_id="3_1_auroc").show()
grouped_bar(leaderboard["model"].tolist(),
            {"Brier": leaderboard["Brier"].tolist(), "ECE": leaderboard["ECE"].tolist()},
            title="Probabilistic quality: Brier and ECE (lower is better)", ylabel="Score (–)",
            save=SAVE_FIGS, fig_id="3_1_brier_ece").show()

[save_figure] PNG для '3_1_leaderboard_rmse' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_1_auroc' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_1_brier_ece' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## ROC curves

In [9]:
y_true = test["label"].cpu().numpy()
series = []
for disp, out in predictions.items():
    fpr, tpr, _ = roc_curve(y_true, out["risk_prob"])
    series.append({"x": fpr, "y": tpr, "name": disp})
series.append({"x": [0, 1], "y": [0, 1], "name": "random", "color": "#1f2937", "dash": "dash", "width": 1.4})
lines(series, title="ROC curves", xlabel="False positive rate (–)", ylabel="True positive rate (–)",
      save=SAVE_FIGS, fig_id="3_1_roc_curves").show()

[save_figure] PNG для '3_1_roc_curves' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Risk calibration

In [10]:
curves = {}
for disp in sample_tables:
    st = sample_tables[disp]
    if st["liq_label"].nunique() > 1:
        frac_pos, mean_pred = calibration_curve(st["liq_label"], st["risk_prob_pred"], n_bins=10)
        curves[disp] = (mean_pred, frac_pos)
calibration_plot(curves, title="Reliability (calibration) curves",
                 save=SAVE_FIGS, fig_id="3_1_calibration").show()

[save_figure] PNG для '3_1_calibration' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Post-hoc temperature scaling

A single temperature T is fitted on the validation set per model and applied to the test
risk logits. This is a fair, universal post-hoc calibration step — it improves Brier/ECE
without changing AUROC (ranking is preserved).

In [11]:
from liquefaction_ai.evaluation import fit_temperature, apply_temperature, expected_calibration_error, safe_binary_metrics

val = benchmark["val"]; y_val = val["label"].cpu().numpy(); y_test = test["label"].cpu().numpy()
cal_rows = []
for name in MODEL_NAMES:
    model, hp, _ = load_trained(name); disp = hp["display_name"]
    val_out = collect_outputs(model, val, config, device)
    vp = np.clip(val_out["risk_prob"], 1e-6, 1 - 1e-6); v_logit = np.log(vp / (1 - vp))
    T = fit_temperature(v_logit, y_val); T = float(np.clip(T if np.isfinite(T) else 1.0, 0.05, 20.0))
    p_raw = np.clip(np.nan_to_num(predictions[disp]["risk_prob"], nan=0.5), 1e-6, 1 - 1e-6)
    p_cal = np.clip(np.nan_to_num(apply_temperature(p_raw, T), nan=0.5), 1e-6, 1 - 1e-6)
    _, _, brier_raw = safe_binary_metrics(y_test, p_raw); ece_raw = expected_calibration_error(y_test, p_raw)
    _, _, brier_cal = safe_binary_metrics(y_test, p_cal); ece_cal = expected_calibration_error(y_test, p_cal)
    cal_rows.append({"Model": disp, "T": round(T, 2), "Brier raw": round(brier_raw, 4), "Brier cal": round(brier_cal, 4),
                     "ECE raw": round(ece_raw, 4), "ECE cal": round(ece_cal, 4)})
cal_df = pd.DataFrame(cal_rows)
display(cal_df)
grouped_bar(cal_df["Model"].tolist(), {"Brier (raw)": cal_df["Brier raw"].tolist(), "Brier (calibrated)": cal_df["Brier cal"].tolist()},
            title="Brier score before/after temperature scaling (lower is better)", ylabel="Brier (–)",
            save=SAVE_FIGS, fig_id="3_1_temperature_scaling").show()

,Model,T,Brier raw,Brier cal,ECE raw,ECE cal
0,MLP-Risk,1.18,0.0033,0.0044,0.0190,0.0259
1,GRU,0.17,0.2047,0.1579,0.1681,0.2728
2,TCN,0.30,0.2180,0.1981,0.1155,0.1561
3,LSTM,0.05,0.2354,0.1412,0.2143,0.2521
4,Transformer,0.16,0.0664,0.0201,0.2127,0.0469
5,FT-Transformer,0.05,0.0668,0.0694,0.2088,0.0741
6,PINN,0.34,0.0433,0.0121,0.1534,0.0368
7,DeepState,0.45,0.1900,0.1811,0.2964,0.2956
8,RealNVP,0.05,0.1936,0.1073,0.2724,0.1098
9,Neural Spline Flow,0.06,0.1478,0.0180,0.3577,0.0265


[save_figure] PNG для '3_1_temperature_scaling' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Uncertainty: coverage and interval width

In [12]:
cov_df = leaderboard.dropna(subset=["Coverage_90"])[["model", "Coverage_90", "Interval_Width_90"]]
display(english_metric_table(cov_df).round(4))
fig = grouped_bar(cov_df["model"].tolist(),
                  {"Coverage@90%": cov_df["Coverage_90"].tolist(),
                   "Interval width@90%": cov_df["Interval_Width_90"].tolist()},
                  title="90% prediction-interval coverage and width", ylabel="Value (–)",
                  save=False, fig_id="3_1_coverage")
fig.add_hline(y=0.90, line_dash="dot", line_color="#dc3545")
from liquefaction_ai.viz import save_figure
save_figure(fig, "3_1_coverage", save=SAVE_FIGS)
fig.show()

,Model,Coverage@90%,Interval width@90%
0,DPI-EVT,0.9356,0.2905
1,PINN,0.9527,0.4014
2,DPI-Flow,0.9194,0.2819
3,Transformer,0.9524,0.4832
4,EVT-NeuralSSM,0.9129,0.4323
5,GRU,0.9553,0.8430
6,Neural Spline Flow,0.7419,0.7225
7,LSTM,1.0000,1.8238
8,RealNVP,0.7218,0.7788
9,TCN,0.9925,2.0765


[save_figure] PNG для '3_1_coverage' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

Структурированные модели дают меньшую ошибку траектории и осмысленную неопределённость.
Дальше — **3.2 абляции и OOD**.